# Argument_Analysis_Agentic-4-capstone : capstone d'integration (baseline 0-shot vs pipeline)

**Navigation** : [<< 3-orchestration](./Argument_Analysis_Agentic-3-orchestration.ipynb) | [README](./README.md) | [UI_configuration >>](./Argument_Analysis_UI_configuration.ipynb)

| Champ | Valeur |
|-------|--------|
| **Module** | SymbolicAI / Argument_Analysis |
| **Rung** | 4-capstone (Epic #2137 Phase 4, capstone d'integration) |
| **Niveau** | Intermediaire |
| **Duree estimee** | 50 minutes |
| **Kernel** | Python 3 (stdlib uniquement) |

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :

- [ ] Componer plusieurs methodes d'analyse **independantes** et identifier les **verdicts convergents** (findings signales par au moins 2 methodes),
- [ ] Implementer les **value-gates VG-1 a VG-4** (FB-18) qui mesurent si une synthese est *groundee* (cite ses artefacts) ou *boilerplate*,
- [ ] Comparer une **baseline 0-shot** (synthese template sans grounding) a un **pipeline integral** (synthese groundee) sur ces gates,
- [ ] Justifier **pourquoi** le pipeline bat la baseline : la valeur se mesure au *grounding* (citations tracables + convergence multi-methode), pas au volume de sortie.

### Prerequis

- Rung [1-informal](./Argument_Analysis_Agentic-1-informal.ipynb) (detection de sophismes) et [3-orchestration](./Argument_Analysis_Agentic-3-orchestration.ipynb) (composition de phases).
- Python intermediaire (dict, regex, decomposition en phrases).

> **Note anti-theatre** : ce capstone distille deux concepts REELS de la production EPITA — les `ConvergentVerdict` (Track DD #637, `agents/core/synthesis/deep_synthesis_models.py`) et les value-gates VG-1..VG-4 (FB-18, `validate_value_gates` dans `agents/core/synthesis/deep_synthesis_agent.py`). La logique deterministe des gates est reproduite fidelement (comptage de citations, mots, paragraphes). En production, la *prose* de synthese est generee par un LLM via `semantic_kernel` ; ici elle est produite deterministiquement pour rester executable sans LLM ni sous-module.

> **Note confidentialite** : le texte analyse est **entierement synthetique** et neutre (comite abstrait, pas d'entite reelle). Aucun corpus EPITA n'est utilise. Le depot est public.


## 1. Introduction : pourquoi un capstone d'integration ?

Les rungs precedents ont construit des briques isolees : un detecteur de sophismes (rung 1), un solveur formel Tweety (rung 2), deux paradigmes d'orchestration (rung 3). Le capstone pose la question qui donne son sens a l'ensemble :

> **Un pipeline integre apporte-t-il une valeur mesurable par rapport a une analyse naive (0-shot), et comment la mesurer ?**

La reponse courte : la valeur d'un pipeline d'analyse argumentative ne se mesure **pas** au volume de texte produit, mais a sa **tracabilite** — la synthese cite-t-elle les artefacts qu'elle invoque ? Plusieurs methodes independantes convergent-elles sur les memes findings ? Ce notebook rend cette intuition **executable** via deux instruments EPITA :

- **ConvergentVerdict** : un finding (argument ou sophisme) signale independamment par au moins 2 methodes est un verdict de haute confiance ; un finding d'une seule methode reste faible.
- **Value-gates VG-1 a VG-4** : quatre gardes deterministes qui distinguent une synthese *groundee* (qui cite ses preuves) d'un *boilerplate* (template vide).

### Ce que ce rung fait (et ne fait pas)

- **Fait** : 3 methodes deterministes sur un texte synthetique, calcul de convergence, 4 value-gates, comparaison baseline vs pipeline.
- **Ne fait pas** : appel LLM (la prose de synthese est deterministe), solveur formel Tweety (rung 2 couvre cela), reseau d'agents `semantic_kernel` (rung 3 couvre l'orchestration).


## 2. Le texte synthetique de reference

Texte argumentatif neutre (comite abstrait, pas d'entite reelle) contenant plusieurs sophismes detectables : appel a l'autorite, attaque ad hominem, generalisation hative, appel a la peur, appel a la conformite. Il sert de **terrain commun** aux methodes, a la baseline et au pipeline.


In [1]:
# Texte synthetique neutre (privacy : aucune entite reelle).
TEXT = (
    "Le comite affirme que le nouveau reglement est necessaire pour la securite. "
    "En effet, tout le monde sait que les reglements previennent les accidents. "
    "D ailleurs, le directeur lui-meme a valide cette decision, donc elle est forcement bonne. "
    "Ceux qui s y opposent n y comprennent rien et cherchent simplement a semer le trouble. "
    "D ailleurs, puisqu aucun accident n a eu lieu cette annee, le reglement n est pas si urgent. "
    "Mais si nous ne l adoptons pas immediatement, la catastrophe est certaine. "
    "Tous les autres comites ont deja adopte un reglement similaire, nous devons faire de meme."
)


def split_sentences(text):
    """Decoupe un texte en phrases (heuristique simple sur '. ').'"""
    parts = [p.strip() for p in text.split(". ") if p.strip()]
    return [p if p.endswith(".") else p + "." for p in parts]


SENTENCES = split_sentences(TEXT)
print(f"{len(SENTENCES)} phrases :")
for i, s in enumerate(SENTENCES, 1):
    print(f"  S{i}: {s}")


7 phrases :
  S1: Le comite affirme que le nouveau reglement est necessaire pour la securite.
  S2: En effet, tout le monde sait que les reglements previennent les accidents.
  S3: D ailleurs, le directeur lui-meme a valide cette decision, donc elle est forcement bonne.
  S4: Ceux qui s y opposent n y comprennent rien et cherchent simplement a semer le trouble.
  S5: D ailleurs, puisqu aucun accident n a eu lieu cette annee, le reglement n est pas si urgent.
  S6: Mais si nous ne l adoptons pas immediatement, la catastrophe est certaine.
  S7: Tous les autres comites ont deja adopte un reglement similaire, nous devons faire de meme.


## 3. Trois methodes d'analyse independantes

Chaque methode produit une liste de *findings*, chacun rattache a l'index de la phrase concernee (`sentence_index`) et porte un `artifact_id`. Ce rattachement est la cle : c'est ce qui permet ensuite de detecter la **convergence** (plusieurs methodes sur la meme phrase).

- **M1** : detecteur de sophismes par lexique (herite du rung 1).
- **M2** : analyse structurelle (affirmation forte sans connecteur de premisse).
- **M3** : detecteur d'absolutisme (mots marquant une certitude exageree).


In [2]:
# M1 : detecteur de sophismes par lexique (rung 1 style).
FALLACY_KEYWORDS = {
    "appel_autorite": ["directeur", "valide cette", "forcement bonne"],
    "ad_hominem": ["comprennent rien", "semer le trouble"],
    "generalisation_hative": ["tout le monde sait", "tout le monde"],
    "appel_peur": ["catastrophe", "certaine"],
    "appel_conformite": ["tous les autres", "faire de meme"],
}


def m1_keyword_detector(sentences):
    findings = []
    fid = 1
    for idx, s in enumerate(sentences):
        s_low = s.lower()
        for family, kws in FALLACY_KEYWORDS.items():
            for kw in kws:
                if kw in s_low:
                    findings.append({
                        "method": "M1_keyword",
                        "artifact_id": f"fall_{fid}",
                        "kind": "sophisme",
                        "family": family,
                        "evidence": kw,
                        "sentence_index": idx,
                    })
                    fid += 1
                    break  # une famille par (phrase, match principal)
    return findings


m1_results = m1_keyword_detector(SENTENCES)
print(f"M1 : {len(m1_results)} findings")
for f in m1_results:
    print(f"  S{f['sentence_index']+1} [{f['family']}] evidence='{f['evidence']}' ({f['artifact_id']})")


M1 : 5 findings
  S2 [generalisation_hative] evidence='tout le monde sait' (fall_1)
  S3 [appel_autorite] evidence='directeur' (fall_2)
  S4 [ad_hominem] evidence='comprennent rien' (fall_3)
  S6 [appel_peur] evidence='catastrophe' (fall_4)
  S7 [appel_conformite] evidence='tous les autres' (fall_5)


In [3]:
# M2 : analyse structurelle -- affirmation forte SANS connecteur de premisse.
CLAIM_WORDS = ["necessaire", "bonne", "urgent", "certaine"]
PREMISE_CONNECTORS = ["donc", "car ", "puisque", "parce que", "en effet"]


def m2_structural_check(sentences):
    findings = []
    fid = 1
    for idx, s in enumerate(sentences):
        s_low = s.lower()
        has_claim = any(w in s_low for w in CLAIM_WORDS)
        has_premise = any(c in s_low for c in PREMISE_CONNECTORS)
        if has_claim and not has_premise:
            findings.append({
                "method": "M2_structural",
                "artifact_id": f"struct_{fid}",
                "kind": "assertion_non_fondee",
                "family": "assertion_non_fondee",
                "evidence": next(w for w in CLAIM_WORDS if w in s_low),
                "sentence_index": idx,
            })
            fid += 1
    return findings


m2_results = m2_structural_check(SENTENCES)
print(f"M2 : {len(m2_results)} findings (affirmations sans premisse)")
for f in m2_results:
    print(f"  S{f['sentence_index']+1} assertion sur '{f['evidence']}' ({f['artifact_id']})")


M2 : 3 findings (affirmations sans premisse)
  S1 assertion sur 'necessaire' (struct_1)
  S5 assertion sur 'urgent' (struct_2)
  S6 assertion sur 'certaine' (struct_3)


In [4]:
# M3 : detecteur d'absolutisme (certitude exageree).
ABSOLUTISM_MARKERS = ["tout le monde", "forcement", "aucun", "certaine", "tous les"]


def m3_absolutism_lens(sentences):
    findings = []
    fid = 1
    for idx, s in enumerate(sentences):
        s_low = s.lower()
        for marker in ABSOLUTISM_MARKERS:
            if marker in s_low:
                findings.append({
                    "method": "M3_absolutism",
                    "artifact_id": f"abs_{fid}",
                    "kind": "absolutisme",
                    "family": "absolutisme",
                    "evidence": marker,
                    "sentence_index": idx,
                })
                fid += 1
                break
    return findings


m3_results = m3_absolutism_lens(SENTENCES)
print(f"M3 : {len(m3_results)} findings (marqueurs d'absolutisme)")
for f in m3_results:
    print(f"  S{f['sentence_index']+1} marqueur='{f['evidence']}' ({f['artifact_id']})")


M3 : 5 findings (marqueurs d'absolutisme)
  S2 marqueur='tout le monde' (abs_1)
  S3 marqueur='forcement' (abs_2)
  S5 marqueur='aucun' (abs_3)
  S6 marqueur='certaine' (abs_4)
  S7 marqueur='tous les' (abs_5)


In [5]:
# Smoke test : les 3 methodes tournent et produisent des findings sur le MEME texte.
METHODS = {"M1_keyword": m1_results, "M2_structural": m2_results, "M3_absolutism": m3_results}
total = sum(len(v) for v in METHODS.values())
print(f"Total findings (3 methodes) : {total}")
for name, findings in METHODS.items():
    print(f"  {name}: {len(findings)}")


Total findings (3 methodes) : 13
  M1_keyword: 5
  M2_structural: 3
  M3_absolutism: 5


### Interpretation : trois perspectives complementaires

Chaque methode regarde le texte sous un angle different : lexicale (M1), structurelle (M2), stylistique (M3). Aucune n'est exhaustive. La question devient : **ou ces perspectives se recoupent-elles ?** Ce recoupement est exactement ce que capture le `ConvergentVerdict`.


## 4. ConvergentVerdict : la convergence multi-methode (Track DD #637)

Un finding signale par **au moins 2 methodes independantes** sur la meme phrase devient un *verdict convergent* : sa confiance est haute parce que deux perspectives distinctes independentment l'ont signale. La `convergence_strength` = nombre de methodes distinctes qui convergent. Les verdicts sont tries par force decroissante (les plus convergents d'abord).

> Distillation fidele de `ConvergentVerdict` (`agents/core/synthesis/deep_synthesis_models.py:111`). En production, la convergence s'opere sur des findings produits par des agents LLM ; ici sur des methodes deterministes.


In [6]:
from collections import defaultdict


def compute_convergent_verdicts(methods_results):
    """Regroupe les findings par phrase, ne garde que ceux ou >= 2 methodes convergent."""
    by_sentence = defaultdict(list)
    for method, findings in methods_results.items():
        for f in findings:
            by_sentence[f["sentence_index"]].append((method, f))

    verdicts = []
    vid = 1
    for sent_idx, items in sorted(by_sentence.items()):
        methods_set = sorted({m for m, _ in items})
        if len(methods_set) >= 2:
            sentence = SENTENCES[sent_idx]
            verdicts.append({
                "verdict_id": f"conv_{vid}",
                "statement": (
                    f"Convergence sur S{sent_idx+1} : {len(methods_set)} methodes "
                    f"({', '.join(methods_set)}) signalent un probleme."
                ),
                "convergence_strength": len(methods_set),
                "sentence_index": sent_idx,
                "sentence": sentence,
                "methods": methods_set,
                "findings": [f for _, f in items],
            })
            vid += 1
    # Tri par force de convergence decroissante.
    verdicts.sort(key=lambda v: -v["convergence_strength"])
    return verdicts


convergent_verdicts = compute_convergent_verdicts(METHODS)
print(f"{len(convergent_verdicts)} verdict(s) convergent(s) (force >= 2), tries par force :")
for v in convergent_verdicts:
    print(f"  [{v['verdict_id']}] force={v['convergence_strength']} | {v['statement']}")
    print(f"        phrase : \"{v['sentence']}\"")


5 verdict(s) convergent(s) (force >= 2), tries par force :
  [conv_4] force=3 | Convergence sur S6 : 3 methodes (M1_keyword, M2_structural, M3_absolutism) signalent un probleme.
        phrase : "Mais si nous ne l adoptons pas immediatement, la catastrophe est certaine."
  [conv_1] force=2 | Convergence sur S2 : 2 methodes (M1_keyword, M3_absolutism) signalent un probleme.
        phrase : "En effet, tout le monde sait que les reglements previennent les accidents."
  [conv_2] force=2 | Convergence sur S3 : 2 methodes (M1_keyword, M3_absolutism) signalent un probleme.
        phrase : "D ailleurs, le directeur lui-meme a valide cette decision, donc elle est forcement bonne."
  [conv_3] force=2 | Convergence sur S5 : 2 methodes (M2_structural, M3_absolutism) signalent un probleme.
        phrase : "D ailleurs, puisqu aucun accident n a eu lieu cette annee, le reglement n est pas si urgent."
  [conv_5] force=2 | Convergence sur S7 : 2 methodes (M1_keyword, M3_absolutism) signalent un prob

### Interpretation : la force de convergence comme indicateur de confiance

Les verdicts convergents sont les findings les plus fiables : ils ne dependent pas d'une seule heuristique (qui pourrait avoir un faux positif). Dans le capstone, une phrase qui cumule un sophisme lexicale (M1), une affirmation sans premisse (M2) **et** un absolutisme (M3) atteint la force maximale 3 — c'est le signal le plus fort que le pipeline peut emettre. Les findings d'une seule methode restent des hypotheses faibles, non elevees au rang de verdict.


## 5. Value-gates VG-1 a VG-4 (FB-18, `validate_value_gates`)

Les value-gates mesurent la qualite de la **synthese generee** (le rapport final). Elles distinguent une synthese *groundee* — qui cite explicitement les artefacts qu'elle invoque via des citations `[artifact:champ:id]` — d'un *boilerplate* (template vide). Reproduction fidele de la logique deterministe EPITA :

| Gate | Condition de reussite | Ce qu'elle detecte |
|------|----------------------|--------------------|
| **VG-1** citation_density | synthese non vide ET nb_citations >= max(1, mots // 200) | synthese qui ne cite pas assez ses sources |
| **VG-2** state_guard | >= 3 champs artefact peuples dans l'etat | run qui n'a rien produit de substantiel |
| **VG-3** no_boilerplate | synthese non vide ET >= 1 citation ET les 4 sections requises presentes | template sans contenu reel |
| **VG-4** transversal_insight | >= 1 paragraphe cite >= 2 champs artefact DISTINCTS | enumeration sans vraie synthese cross-artefact |


In [7]:
import re

# Pattern de citation fidele a EPITA : [artifact:<champ>:<id>]
CITATION_RE = re.compile(r"\[artifact:([a-z_]+):?(\d*)\]")

# Les 4 sections requises dans une synthese groundee.
REQUIRED_SECTIONS = [
    "## Arguments identifies",
    "## Sophismes detectes",
    "## Verdicts convergents",
    "## Synthese",
]


def validate_value_gates(synthesis_md, populated_artifact_fields):
    """VG-1..VG-4 sur une synthese groundee (FB-18, logique EPITA fidele)."""
    citations = CITATION_RE.findall(synthesis_md or "")
    n_citations = len(citations)
    distinct_fields = sorted({fn for fn, _ in citations})
    words = len((synthesis_md or "").split())
    required_citations = max(1, words // 200)

    vg1 = bool(synthesis_md) and n_citations >= required_citations
    vg2 = populated_artifact_fields >= 3
    vg3 = (
        bool(synthesis_md)
        and n_citations >= 1
        and all(h in synthesis_md for h in REQUIRED_SECTIONS)
    )
    vg4 = False
    for para in (synthesis_md or "").split("\n\n"):
        para_fields = {fn for fn, _ in CITATION_RE.findall(para)}
        if len(para_fields) >= 2:
            vg4 = True
            break

    return {
        "vg1_citation_density": {
            "pass": vg1, "citations": n_citations, "words": words,
            "required_citations": required_citations, "distinct_fields": distinct_fields,
        },
        "vg2_state_guard": {
            "pass": vg2, "fields": populated_artifact_fields, "required": 3,
        },
        "vg3_no_boilerplate": {"pass": vg3},
        "vg4_transversal_insight": {"pass": vg4},
        "all_pass": vg1 and vg2 and vg3 and vg4,
    }


# Smoke test sur une citation isolee.
_demo = "La phrase [artifact:arguments:1] est faible [artifact:sophismes:2]."
_demo_vg = validate_value_gates(_demo, 2)
print("Smoke value-gates (citation isolee, 2 champs) :")
for k in ("vg1_citation_density", "vg2_state_guard", "vg3_no_boilerplate", "vg4_transversal_insight"):
    print(f"  {k}: pass={_demo_vg[k]['pass']}")
print(f"  all_pass={_demo_vg['all_pass']} (VG-2 attendu ECHOUC : 2 < 3 champs)")


Smoke value-gates (citation isolee, 2 champs) :
  vg1_citation_density: pass=True
  vg2_state_guard: pass=False
  vg3_no_boilerplate: pass=False
  vg4_transversal_insight: pass=True
  all_pass=False (VG-2 attendu ECHOUC : 2 < 3 champs)


## 6. Baseline 0-shot vs pipeline integral

C'est le coeur du capstone. Deux syntheses sont generees pour le meme texte :

- **Baseline 0-shot** : un template generique avec les 4 sections requises mais **sans aucune citation** et un etat a peine peuple (1 champ). C'est ce que produirait un generateur qui n'exploite pas les methodes.
- **Pipeline integral** : compose les 3 methodes, peuple un etat avec 3+ champs artefact (arguments, sophismes, verdicts convergents) et genere une synthese qui **cite** ces artefacts.

Les value-gates tranchent : la baseline doit echouer, le pipeline doit reussir.


In [8]:
def baseline_synthesis():
    """Template 0-shot : sections presentes mais AUCUNE citation, etat vide."""
    synth = (
        "## Arguments identifies\n\n"
        "Le texte presente plusieurs affirmations.\n\n"
        "## Sophismes detectes\n\n"
        "Une analyse plus approfondie est necessaire.\n\n"
        "## Verdicts convergents\n\n"
        "A determiner.\n\n"
        "## Synthese\n\n"
        "Ce texte argumentatif merite une etude detaillee. "
        "Les points principaux restent a confirmer par une analyse approfondie."
    )
    # Etat a peine peuple : seul le champ "synthese_boilerplate".
    populated_fields = 1
    return synth, populated_fields


baseline_synth, baseline_fields = baseline_synthesis()
baseline_vg = validate_value_gates(baseline_synth, baseline_fields)
print("=== Baseline 0-shot ===")
print(f"champs peuples : {baseline_fields}")
for k in ("vg1_citation_density", "vg2_state_guard", "vg3_no_boilerplate", "vg4_transversal_insight"):
    print(f"  {k}: pass={baseline_vg[k]['pass']}")
print(f"  all_pass={baseline_vg['all_pass']}")


=== Baseline 0-shot ===
champs peuples : 1
  vg1_citation_density: pass=False
  vg2_state_guard: pass=False
  vg3_no_boilerplate: pass=False
  vg4_transversal_insight: pass=False
  all_pass=False


In [9]:
def pipeline_synthesis(sentences, methods_results, verdicts):
    """Pipeline integral : synthese GROUNDEE qui cite les artefacts.'"""
    # Extraction simple : chaque phrase est un candidat-argument.
    arguments = [{"id": f"arg_{i+1}", "text": s} for i, s in enumerate(sentences)]

    # Toutes les findings de sophismes (toutes methodes confondues).
    fallacies = []
    for method, findings in methods_results.items():
        for f in findings:
            fallacies.append(f)

    # Etat peuple : 3+ champs artefact distincts.
    populated_fields = 3  # arguments, sophismes, convergent_verdicts

    parts = []
    parts.append("## Arguments identifies\n")
    for a in arguments:
        parts.append(f"- Argument [{a['id']}] [artifact:arguments:{a['id'].split('_')[1]}] : {a['text']}")
    parts.append("\n## Sophismes detectes\n")
    for i, f in enumerate(fallacies, 1):
        parts.append(f"- {f['family']} (S{f['sentence_index']+1}) [artifact:sophismes:{i}] : par {f['method']}.")
    parts.append("\n## Verdicts convergents\n")
    for i, v in enumerate(verdicts, 1):
        parts.append(f"- {v['statement']} [artifact:convergent:{i}] (force {v['convergence_strength']}).")
    parts.append("\n## Synthese\n")
    # Paragraphe de synthese citant >= 2 champs distincts -> VG-4.
    parts.append(
        f"Le pipeline identifie {len(arguments)} arguments [artifact:arguments:1] et "
        f"{len(fallacies)} sophismes [artifact:sophismes:1]. La convergence est forte : "
        f"{len(verdicts)} verdicts de force >= 2 [artifact:convergent:1], dont le plus solide "
        f"atteint la force {verdicts[0]['convergence_strength'] if verdicts else 0}. "
        f"Ces recoupements confirment un raisonnement fragile [artifact:arguments:2]."
    )
    return "\n".join(parts), populated_fields


pipeline_synth, pipeline_fields = pipeline_synthesis(SENTENCES, METHODS, convergent_verdicts)
pipeline_vg = validate_value_gates(pipeline_synth, pipeline_fields)
print("=== Pipeline integral ===")
print(f"champs peuples : {pipeline_fields}")
for k in ("vg1_citation_density", "vg2_state_guard", "vg3_no_boilerplate", "vg4_transversal_insight"):
    print(f"  {k}: pass={pipeline_vg[k]['pass']}")
print(f"  all_pass={pipeline_vg['all_pass']}")


=== Pipeline integral ===
champs peuples : 3
  vg1_citation_density: pass=True
  vg2_state_guard: pass=True
  vg3_no_boilerplate: pass=True
  vg4_transversal_insight: pass=True
  all_pass=True


In [10]:
# Tableau comparatif baseline vs pipeline.
print(f"{'Gate':<28} {'Baseline 0-shot':<18} {'Pipeline integral':<18}")
print("-" * 64)
order = ("vg1_citation_density", "vg2_state_guard", "vg3_no_boilerplate", "vg4_transversal_insight")
for k in order:
    b = "REUSSI" if baseline_vg[k]["pass"] else "ECHOUC"
    p = "REUSSI" if pipeline_vg[k]["pass"] else "ECHOUC"
    print(f"{k:<28} {b:<18} {p:<18}")
print("-" * 64)
b_all = "REUSSI" if baseline_vg["all_pass"] else "ECHOUC"
p_all = "REUSSI" if pipeline_vg["all_pass"] else "ECHOUC"
print(f"{'all_pass':<28} {b_all:<18} {p_all:<18}")
print()
print(f"Verdicts convergents (pipeline) : {len(convergent_verdicts)}, "
      f"force max = {max((v['convergence_strength'] for v in convergent_verdicts), default=0)}")


Gate                         Baseline 0-shot    Pipeline integral 
----------------------------------------------------------------
vg1_citation_density         ECHOUC             REUSSI            
vg2_state_guard              ECHOUC             REUSSI            
vg3_no_boilerplate           ECHOUC             REUSSI            
vg4_transversal_insight      ECHOUC             REUSSI            
----------------------------------------------------------------
all_pass                     ECHOUC             REUSSI            

Verdicts convergents (pipeline) : 5, force max = 3


### Interpretation : la valeur se mesure au grounding

La baseline et le pipeline produisent tous deux un texte avec les 4 sections requises. Pourtant les value-gates les **distinguent nettement** :

- **VG-1** : la baseline ne cite aucun artefact (0 citation) alors qu'elle compte ~60 mots (seuil 1) ; le pipeline cite abondamment.
- **VG-2** : la baseline n'a peuple qu'un champ (template) ; le pipeline en a 3 (arguments, sophismes, verdicts).
- **VG-3** : la baseline a les sections mais **aucune citation** = boilerplate detecte.
- **VG-4** : le pipeline possede un paragraphe (la synthese) qui cite >= 2 champs distincts (arguments + sophismes + convergent) = *transversal insight*, la marque d'une vraie synthese et non d'une enumeration.

La lecon : **la valeur d'un pipeline d'analyse argumentative est la tracabilite**. Une synthese qui dit "le raisonnement est fragile" vaut peu si elle ne pointe pas *quel* argument, *quel* sophisme, et si plusieurs methodes convergent. Les value-gates rendent cette qualite **auditable et automatique**.


### Exercice 1 : ajouter une 5e value-gate (convergence floor)

Ajoutez **VG-5 convergence_floor** : `pass = au moins un ConvergentVerdict de force >= 2`. Integrez-la dans `validate_value_gates` (ou une fonction derivee) et executez-la sur la baseline (etat sans verdicts) et le pipeline. Que conclure sur la capacity d'une baseline a "passer" cette gate ?


In [11]:
# Exercice 1 -- a completer
# TODO etudiant : definir vg5_convergence_floor et l'evaluer sur baseline + pipeline.
# Indice : la baseline n'a pas de verdicts ; le pipeline a `convergent_verdicts`.
# Etape 1 : ecrire la fonction prenant (synthesis_md, populated_fields, verdicts).
# Etape 2 : l'appliquer aux deux syntheses.
pass
print("Exercice 1 a completer : definir vg5_convergence_floor et evaluer sur baseline + pipeline")

Exercice 1 a completer : definir vg5_convergence_floor et evaluer sur baseline + pipeline


### Exercice 2 : diagnostic des gates qui echouent

Ecrivez une fonction `diagnose_failures(vg_result)` qui retourne, pour chaque gate en echec, une **ligne de diagnostic** explicant pourquoi (ex. "VG-1 : 0 citations pour 60 mots, seuil 1"). Testez-la sur la baseline.


In [12]:
# Exercice 2 -- a completer
# TODO etudiant : ecrire diagnose_failures(vg_result) -> List[str].
# Indice : chaque gate porte dans vg_result les compteurs (citations, words, required_citations, fields...).
# Etape 1 : iterer sur vg1..vg4.
# Etape 2 : pour chaque gate not pass, formater un message utilisant ses compteurs.
result = None  # TODO etudiant
print("Exercice 2 a completer : ecrire diagnose_failures(vg_result) -> List[str]")

Exercice 2 a completer : ecrire diagnose_failures(vg_result) -> List[str]


### Exercice 3 : synthese adversariale (citation-stuffing)

Construisez une synthese **adversariale** qui tente de "tromper" les gates en bourrant des citations `[artifact:...]` **sans analyse reelle** (un seul champ artefact repete). Observez quelles gates passent et lesquelles echouent encore. Conclusion : pourquoi VG-4 (transversal insight) est-elle essentielle pour completer VG-1 ?


In [13]:
# Exercice 3 -- a completer
# TODO etudiant : construire adversarial_synth qui bourre des citations d'UN seul champ.
# Indice : VG-1 compte les citations brutes, VG-4 exige >= 2 champs DISTINCTS par paragraphe.
# Etape 1 : ecrire la synthese avec beaucoup de [artifact:arguments:N] mais jamais 2 champs differents.
# Etape 2 : la soumettre a validate_value_gates et observer.
pass
print("Exercice 3 a completer : construire adversarial_synth (citations d'un seul champ)")

Exercice 3 a completer : construire adversarial_synth (citations d'un seul champ)


## 7. Pont vers l'état partagé

Les verdicts convergents et les findings des trois méthodes alimentent le même **conteneur partagé** — l'objet `RhetoricalAnalysisState` du module `argumentation_lib` — que **tous les rungs** enrichissent. Le rung 1 y écrit ses détections informelles, le rung 2 y ajoute les verdicts formels de Tweety, le rung 3 y dépose le résultat de l'orchestration, et ce capstone y verse ses **verdicts convergents** : le signal le plus riche de la série.

Cette section montre le geste d'intégration final : on mappe les findings canoniques du pipeline (sophismes de M1) et la force de convergence vers le conteneur partagé. C'est l'objet qu'une UI ou un rapport de production lirait pour combiner informel (rung 1), formel (rung 2), orchestration (rung 3) et convergence capstone (rung 4).

> Le conteneur est **déterministe** (un dictionnaire typé sans logique LLM). Le remplir ne coûte aucun appel API : on réutilise les `convergent_verdicts` et les `m1_results` déjà calculés ci-dessus. En production, c'est ce même conteneur que `conversational_orchestrator.py` (Semantic Kernel) remplit via des agents LLM — la forme de l'objet est identique, seuls les producteurs changent.

In [14]:
# Pont vers l'etat partage : on verse les verdicts convergents du capstone
# dans RhetoricalAnalysisState, le conteneur commun a tous les rungs. Rung 1 y
# ecrit ses detections informelles, rung 2 y ajoute les verdicts formels de
# Tweety, rung 3 y depose le resultat orchestre, et ce capstone (4) y verse
# ses verdicts convergents -- le signal le plus riche de la serie. Aucun LLM
# requis : on reutilise les sorties deja calculees ci-dessus.

import logging

# Le package argumentation_lib vit a cote du notebook ; il est importable
# directement (le dossier du notebook est sur sys.path au runtime). La shim
# resout ses ressources (data/, libs/) via _paths.py (__file__-relative, donc
# cwd-independant). Aucun guard de path ni raise dans la cellule (regle C.1) :
# la robustesse cwd est portee par la shim, comme aux rungs 1/2/3.
from argumentation_lib import RhetoricalAnalysisState

logging.getLogger("RhetoricalAnalysisState").setLevel(logging.WARNING)

# Remplissage du conteneur partage : un argument cible agregeant le bloc
# argumentatif, puis un sophisme canonique (M1) par phrase detectee, avec la
# force de convergence en bonus dans la justification quand la phrase est un
# verdict convergent (force >= 2).
etat_partage = RhetoricalAnalysisState(initial_text=TEXT)
arg_id = etat_partage.add_argument(
    "Bloc argumentatif synthetique (7 phrases analysees par le capstone)."
)
for f in m1_results:
    force = 0
    for v in convergent_verdicts:
        if v["sentence_index"] == f["sentence_index"]:
            force = v["convergence_strength"]
            break
    justification = (
        f"declencheur M1 : '{f['evidence']}' (phrase S{f['sentence_index']+1})"
    )
    if force >= 2:
        justification += f" ; verdict convergent de force {force}"
    etat_partage.add_fallacy(
        fallacy_type=f["family"],
        justification=justification,
        target_arg_id=arg_id,
        family=f["family"],
    )

snapshot = etat_partage.get_state_snapshot(summarize=True)

print("--- Pont vers l'etat partage (argumentation_lib) ---")
print(f"arguments dans l etat : {snapshot['argument_count']}")
print(f"sophismes dans l etat : {snapshot['fallacy_count']}")
print(f"verdicts convergents   : {len(convergent_verdicts)} (force max {max((v['convergence_strength'] for v in convergent_verdicts), default=0)})")
print(f"texte brut (extrait)   : {snapshot['raw_text_snippet'][:60]}...")
print("Ce conteneur combine informel (rung 1), formel (rung 2), orchestration (rung 3) et convergence (rung 4).")

--- Pont vers l'etat partage (argumentation_lib) ---
arguments dans l etat : 1
sophismes dans l etat : 5
verdicts convergents   : 5 (force max 3)
texte brut (extrait)   : Le comite affirme que le nouveau reglement est necessaire po...
Ce conteneur combine informel (rung 1), formel (rung 2), orchestration (rung 3) et convergence (rung 4).


## 8. Conclusion et recapitulatif

Ce capstone ferme la serie en montrant que l'interet d'un pipeline d'analyse argumentative **se mesure**, pas se se declame. Deux instruments EPITA rendent la mesure executable :

### Tableau recapitulatif

| Instrument | Origine EPITA | Role dans le capstone |
|------------|---------------|----------------------|
| `ConvergentVerdict` | Track DD #637 (`deep_synthesis_models.py`) | Force de confiance = nb de methodes qui convergent sur une meme phrase |
| `validate_value_gates` VG-1..VG-4 | FB-18 (`deep_synthesis_agent.py`) | Distinguent une synthese groundee d'un boilerplate |

### Points a retenir

1. **La convergence multi-methode est un signal de confiance** : un finding confirme par 2+ methodes independantes est plus fiable qu'un finding isole.
2. **Une synthese groundee cite ses artefacts** : les citations `[artifact:...]` rendent le raisonnement auditable, pas juste plausible.
3. **Les value-gates composent** : VG-1 (densite) + VG-4 (transversalite) empechent de "tromper" le systeme par citation-stuffing — il faut aussi de la diversite cross-artefact.
4. **La baseline 0-shot echoue sur les 4 gates** : produire du texte avec des sections ne suffit pas ; il faut du contenu trace.
5. **Pont vers l'etat partage**. Les verdicts convergents du capstone se versent dans le meme conteneur `RhetoricalAnalysisState` que les autres rungs (section 7). C'est le **bus d'integration terminal** : une UI ou un rapport de production n'a qu'a lire cet objet pour combiner informel (rung 1), formel (rung 2), orchestration (rung 3) et convergence (rung 4) en un etat coherent.

### Vers la production

En production EPITA, la prose de synthese est generee par un LLM via `semantic_kernel`, contrainte a ne citer que des artefacts reels (prompt *strictement grounde*). Les value-gates servent alors de **garde-fous CI** : une synthese qui echoue une gate est rejetee automatiquement, evitant les hallucinations. Ce capstone reproduit la *logique* des gates (deterministe) pour la rendre palpable sans LLM.

### Prochaines etapes

- Le rung [0-init](./Argument_Analysis_Agentic-0-init.ipynb) couvre la configuration de l'environnement (JVM Tweety, fail-loud) pour qui veut brancher les vrais solveurs.
- Pour aller plus loin sur la confiance multi-methode : explorer les sémantiques de Dung dans le rung [2-formal](./Argument_Analysis_Agentic-2-formal.ipynb) (acceptabilite graduee).